<a href="https://colab.research.google.com/github/ShikharVeer10/TokenCompression/blob/main/Copy_of_Token_Compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **PHASE 1**

In [1]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU Available: {gpu_name}")
    print(f"✅ Total VRAM: {total_memory:.2f} GB")
else:
    print("❌ No GPU found! Please change your runtime to T4 GPU.")

✅ GPU Available: Tesla T4
✅ Total VRAM: 14.56 GB


In [2]:
!pip install --upgrade pip setuptools wheel
!pip install -q -U "tokenizers>=0.19.1"
!pip install -q -U transformers bitsandbytes accelerate pydantic psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 63.5 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


ERROR: Operation cancelled by user
^C


In [5]:
# Fix: Upgrade huggingface_hub to be compatible with transformers 4.48+
!pip install -q "huggingface_hub>=1.5.0,<2.0"

import transformers
import bitsandbytes
import accelerate

print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.16.1
BitsAndBytes: 0.50.2
Accelerate: 1.14.0


In [6]:
import torch
from transformers import BitsAndBytesConfig

# Define the 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # High-precision 4-bit format
    bnb_4bit_compute_dtype=torch.float16,    # Math happens in float16
    bnb_4bit_use_double_quant=True           # Extra memory compression
)

print("BitsAndBytes 4-bit configuration is ready.")

BitsAndBytes 4-bit configuration is ready.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-7B-Instruct"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Downloading and loading 4-bit base model (takes ~2-4 mins)...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"  # Vital for later attention hooks
)

print("Model successfully loaded into GPU memory!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import time
import torch

def get_vram_usage():
    """Returns allocated and peak VRAM in Gigabytes."""
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    return allocated, peak

def run_baseline_inference(prompt: str, max_new_tokens: int = 128):
    # Reset GPU peak tracking so we measure only this run
    torch.cuda.reset_peak_memory_stats()

    # Tokenize input text
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_token_count = inputs.input_ids.shape[1]

    # Start timer
    start_time = time.time()

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # Deterministic for exact benchmarking
            use_cache=True     # Standard full uncompressed KV-cache
        )

    latency = time.time() - start_time
    output_token_count = outputs.shape[1] - input_token_count
    allocated_vram, peak_vram = get_vram_usage()

    # Decode model output
    response_text = tokenizer.decode(outputs[0][input_token_count:], skip_special_tokens=True)

    return {
        "response": response_text,
        "input_tokens": input_token_count,
        "output_tokens": output_token_count,
        "latency_sec": round(latency, 2),
        "tokens_per_sec": round(output_token_count / latency, 2),
        "peak_vram_gb": round(peak_vram, 2)
    }

print("Baseline testing function created successfully.")

In [ ]:
test_prompt = """<|im_start|>system
You are a concise technical assistant.<|im_end|>
<|im_start|>user
Explain in two sentences what a Key-Value Cache is in Large Language Models.<|im_end|>
<|im_start|>assistant
"""

baseline_results = run_baseline_inference(test_prompt, max_new_tokens=100)

print("--- BASELINE UNCOMPRESSED METRICS ---")
print(f"Prompt Tokens      : {baseline_results['input_tokens']}")
print(f"Generated Tokens   : {baseline_results['output_tokens']}")
print(f"Total Latency      : {baseline_results['latency_sec']} seconds")
print(f"Throughput         : {baseline_results['tokens_per_sec']} tokens/sec")
print(f"Peak VRAM Footprint: {baseline_results['peak_vram_gb']} GB")
print("-" * 36)
print(f"Model Output:\n{baseline_results['response']}")



---

## **PHASE 2**

In [ ]:
!pip install -q llmlingua

In [ ]:
from llmlingua import PromptCompressor

print("Loading LLMLingua-2 model onto GPU...")

compressor = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2=True,
    device_map="cuda"
)

print("LLMLingua-2 loaded successfully!")

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

# 1. Strict Data Contract using Pydantic
class CompressionRequest(BaseModel):
    context: str
    instruction: Optional[str] = Field(default="", description="The query or task instruction")
    rate: float = Field(default=0.5, ge=0.1, le=0.9, description="Target retention rate (0.5 means keep 50% of tokens)")

class CompressionResponse(BaseModel):
    compressed_text: str
    origin_tokens: int
    compressed_tokens: int
    tokens_saved: int
    pruning_percentage: float

# 2. Wrapper function using LLMLingua-2
def compress_context(request: CompressionRequest) -> CompressionResponse:
    # Essential structural punctuation marks to preserve
    protected_tokens = ['\n', '?', '.', ':', '-', '"', "'"]

    results = compressor.compress_prompt(
        context=[request.context],
        instruction=request.instruction,
        rate=request.rate,
        force_tokens=protected_tokens
    )

    orig = results['origin_tokens']
    comp = results['compressed_tokens']
    saved = orig - comp
    savings_pct = round((saved / orig) * 100, 2) if orig > 0 else 0.0

    return CompressionResponse(
        compressed_text=results['compressed_prompt'],
        origin_tokens=orig,
        compressed_tokens=comp,
        tokens_saved=saved,
        pruning_percentage=savings_pct
    )

print("Compression wrapper created successfully.")

In [ ]:
# A realistic wordy technical scenario
sample_context = """
Dear infrastructure engineering team, I am writing this incident report to formally notify you
that we experienced an unexpected hardware fault in cluster region US-East-4 at approximately
03:45 AM UTC. The secondary cooling distribution unit malfunctioned, causing the server rack
temperatures to surge rapidly from a normal baseline of 42 degrees Celsius up to a critical threshold
of 89 degrees Celsius. Under the high thermal load, node worker-12 experienced memory throttling,
and the distributed cache service threw repeated critical error code 0x884F before shutting down.
Please refer to standard maintenance protocol section 9B to initiate the automated thermal fallback
procedure and redirect network traffic to the secondary standby node immediately.
"""

sample_query = "What was the critical error code, the peak temperature, and which protocol should be followed?"

# Create our structured request targeting a 50% retention rate (rate=0.5)
request = CompressionRequest(
    context=sample_context,
    instruction=sample_query,
    rate=0.5
)

# Run compression
compressed_data = compress_context(request)

print("--- STAGE 1 (LLMLINGUA-2) PRUNING RESULTS ---")
print(f"Original Token Count  : {compressed_data.origin_tokens}")
print(f"Compressed Token Count: {compressed_data.compressed_tokens}")
print(f"Tokens Eliminated     : {compressed_data.tokens_saved}")
print(f"Pruning Percentage    : {compressed_data.pruning_percentage}%")
print("-" * 50)
print("PRUNED TEXT PAYLOAD:")
print(compressed_data.compressed_text)

In [ ]:
# Format the pruned text into our model's chat template
pruned_prompt = f"""<|im_start|>system
You are a precise technical assistant. Answer the question using only the provided context.<|im_end|>
<|im_start|>user
Context:
{compressed_data.compressed_text}

Question:
{sample_query}<|im_end|>
<|im_start|>assistant
"""

# Run inference with the pruned prompt
pruned_results = run_baseline_inference(pruned_prompt, max_new_tokens=100)

print("--- STAGE 1 INFERENCE RESULTS ---")
print(f"Prompt Tokens Feed to Model: {pruned_results['input_tokens']}")
print(f"Generated Answer Tokens    : {pruned_results['output_tokens']}")
print(f"Execution Latency          : {pruned_results['latency_sec']} seconds")
print(f"Peak VRAM Footprint        : {pruned_results['peak_vram_gb']} GB")
print("-" * 50)
print(f"Answer Generated:\n{pruned_results['response']}")

**PHASE-3**

In [2]:
!pip install -U bitsandbytes>=0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import torch
import torch.nn.functional as F
from transformers.cache_utils import DynamicCache
from pydantic import BaseModel,Field,model_validator
from typing import Literal,Optional

In [ ]:
#Pydantic configuration contract
class SnapKVConfig(BaseModel):
  max_capacity:int=Field(default=1024,ge=128,description="Maximum tokens retained in KV cache") #Maximum no of tokens to be allowed in memory before deleting
  window_size:int=Field(default=32,ge=8, description="Recent tokens observation window") #The number of tokens we look to decide what is important
  kernel_size:int=Field(default=5,ge=1, description="Pooling kernel size for attention voting") #This is to cluster attention scores together

  @model_validator(mode="after")
  def validate_capacity(self):
    if self.window_size>=self.max_capacity:
      raise ValueError("Window Size should be smaller than the maximum KV capacity")
    return self


In [ ]:
class ExperimentalRunConfig(BaseModel):
  model_name:Literal["Qwen/Qwen2.5-7B-Instruct", "meta-llama/Meta-Llama-3-8B-Instruct"]
  dataset_name:str="LongBench"
  task_name:str
  condition:Literal["baseline","llmlingua_only","snapkv_only","integrated"]
  max_input_tokens:int=16000

class BenchMetricRecord(BaseModel):
  config:ExperimentalRunConfig
  ttft_ms:float
  throughput_token_per_sec:float
  peak_vram_gb:float
  prompt_compression_ratio:float
  kv_compression_ratio:float
  task_accuracy_f1:Optional[float]=None


In [ ]:

import torch
import torch.nn.functional as F

print("Attaching SnapKV hooks to existing model")

class SnapKVManager:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.hooks = []
        self.num_kv_heads = model.config.num_key_value_heads
        self.num_heads = model.config.num_attention_heads
        self.num_groups = self.num_heads // self.num_kv_heads

    def _attention_hook(self, module, inputs, output):
        # Logic to compress KV cache based on attention importance
        if not isinstance(output, tuple) or len(output) < 3 or output[2] is None: return output
        attention_weights=output[1]
        past_key_value=output[2]
        keys,values=past_key_value[0],past_key_value[1]

        current_memory_size=keys.shape[2] #Total tokens currently stored in memory

        if current_memory_size<=self.config.max_capacity:
          return output

        bsz=keys.shape[0] #Batch size
        seq_len=attention_weights.shape[-1] #Sequence length
        attention_reshaped=attention_weights.view(bsz,self.num_kv_heads,self.num_groups,attention_weights.shape[2],seq_len) #Reshapes the model query attention weights. It groups 28 heads into 7 clusters instead of 4
        kv_attention=attention_reshaped.mean(dim=2)
        recent_attention=kv_attention[:,:, -self.config.window_size:, :-self.config.window_size] #Measures the attention paid to the previous tokens
        historical_scores=recent_attention.mean(dim=-2) #Averages attention scores across the heads
        smoothed_scores=F.avg_pool1d(historical_scores,kernel_size=self.config.kernel_size,stride=1,padding=self.config.kernel_size//2)
        keep_historical=self.config.max_capacity - self.config.window_size #Calculates how many historical token we can manage to keep

        _,top_indices=torch.topk(smoothed_scores,k=keep_historical,dim=1) #Finds the index position
        top_indices,_=torch.sort(top_indices,dim=-1) #Sorts the index position from lowest to highest
        idx_expanded = top_indices.unsqueeze(-1).expand(-1, -1, -1, keys.shape[-1]) #Expands the index map dimensions so it matches the hidden feature size of the model's key/value tensors

        historical_keys=keys[:,:, :-self.config.window_size, :] #Seperates older history from recent observation window
        historical_values=values[:,:, :-self.config.window_size, :]

        retained_keys = torch.gather(historical_keys, 2, idx_expanded)
        retained_values = torch.gather(historical_values, 2, idx_expanded)

        #protected_keys--> Saves and stores the last 32 recent tokens.
        protected_keys=keys[:,:,-self.config.window_size:,:]
        protected_values=values[:,:,-self.config.window_size:,:]

        new_keys = torch.cat([retained_keys, protected_keys], dim=2)
        new_values = torch.cat([retained_values, protected_values], dim=2)

        return (output[0], output[1], (new_keys, new_values))

    #Loops through every transformer layer to the attention blocks
    def turn_on(self):
        for layer in self.model.model.layers:
            hook = layer.self_attn.register_forward_hook(self._attention_hook)
            self.hooks.append(hook)

    #This removes all active hooks returning the model back to normal state
    def turn_off(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

snapkv_engine = SnapKVManager(base_model, SnapKVConfig())
snapkv_engine.turn_on()
print(f"SnapKV hooks attached to {len(base_model.model.layers)} layers.")
results = run_baseline_inference("Explain KV cache compression benefits.", max_new_tokens=50)
print(f"\n SNAPKV METRICS \nLatency: {results['latency_sec']}s\nVRAM: {results['peak_vram_gb']}GB\nResponse: {results['response']}")


In [ ]:
snap_config=SnapKVConfig(max_capacity=512,window_size=32,kernel_size=5)
snapkv_engine=SnapKVManager(base_model,snap_config)
snapkv_engine.turn_on()
print("SnapKV hooks attached to all layers.")

test_prompt= (
    "<|im_start|>user\n"
    "Explain how KV cache memory compression works in large language models "
    "and why it is crucial for handling long-context documents without running out of VRAM.\n"
    "<|im_end|>\n<|im_start|>assistant\n"
)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
print("Generating response with active SnapKV compression")
with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=80, do_sample=False, use_cache=True,output_attentions=True)
    response_text=tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response_text)

    snapkv_engine.turn_off() #Hooks are turned off when testing is done
    print("\n The SnapKV Hooks are detached")

**PHASE-4**

In [ ]:
import time
import math
import torch
from pydantic import BaseModel,Field
from typing import List, Dict

In [ ]:
#Pydantic Schema for Telemetry validation
class BenchmarkRecord(BaseModel):
  configuration:str=Field(description="Baseline, LLMLingua_Only, SnapKV_only, or Integrated_Pipeline")
  prompt_tokens:int=Field(description="Total input tokens fed into the model")
  ttft_sec:float=Field(description="Time-to-First-Token prefill latency in seconds")
  throughput_tps:float=Field(description="Tokens per second during autoregression decoding")
  peak_vram_gb:float=Field(description="Maximum GPU memory consumed in Gigabytes")
  perplexity:float=Field(description="Sequence probability score")
  token_f1:float=Field(description="Token-level F1 match score against ground truth")
  rouge_l:float=Field(description="ROUGE-L structural recall/precision score")

def calculate_f1_rouge(prediction:str, ground_truth:str):
  pred_tokens=prediction.strip().lower().split()
  gt_tokens=ground_truth.strip().lower().split()

  if not pred_tokens or not gt_tokens:
    return 0.0,0.0

  common=set(pred_tokens) & set(gt_tokens)
  precision=len(common)/ len(pred_tokens) if pred_tokens else 0
  recall=len(common)/len(gt_tokens) if gt_tokens else 0
  f1=round((2*precision*recall)/(precision+recall), 4) if (precision + recall)>0 else 0.0

  m,n=len(pred_tokens), len(gt_tokens)
  dp = [[0] * (n+1) for _ in range(m+1)]
  for i in range(1,m+1):
    for j in range(1,n+1):
      if pred_tokens[i-1]==gt_tokens[j-1]:
        dp[i][j]=dp[i-1][j-1]+1
      else:
        dp[i+1][j+1]=max(dp[i+1][j],dp[i][j+1])

  lcs=dp[m][n]
  r_prec=lcs/m if m>0 else 0
  r_rec=lcs/n if n>0 else 0
  rouge_l=round((2*r_prec*r_rec)/(r_prec+r_rec) if (r_prec+r_rec)>0 else 0.0,4)

  return f1,rouge_l

@torch.no_grad()
def calculate_perplexity(model,tokenizer,text:str)->float:
  input_ids=tokenizer(text,return_sensors="pt").to("cuda").input_ids
  if input_ids.shape[1]==0:
    return 0.0
  loss=model(input_ids,labels=input_ids).loss
  return round(math.exp(loss.item()),2)

print("Perplexity calculation function created successfully.")




In [ ]:
%pip install -U datasets

In [ ]:
%pip install -U datasets==2.19.0 huggingface_hub==0.23.0
%pip install -U "datasets>=2.20.0" "huggingface_hub>=0.23.2"

In [ ]:
!pip install -U datasets==2.14.0

In [ ]:
!pip install -q -U "datasets>=2.21.0" "huggingface_hub>=0.24.0,<2.0"

In [ ]:
import sys

# Flush stale modules from memory
for mod in list(sys.modules.keys()):
    if mod.startswith("huggingface_hub") or mod.startswith("datasets"):
        del sys.modules[mod]

import huggingface_hub
from datasets import load_dataset

print("Datasets version:", load_dataset.__module__)
print("Hub version     :", huggingface_hub.__version__)
print("✅ Imports succeeded cleanly!")

In [ ]:
import os
!pip install -q "huggingface_hub>=1.5.0,<2.0" "datasets>=2.20.0"

# Automatically restart the runtime to apply changes to the Python path
print("Done! Restarting runtime to apply version changes. Please wait for reconnection...")
os.kill(os.getpid(), 9)

In [ ]:
import io
import time
import math
import torch
import torch.nn.functional as F
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pydantic import BaseModel, Field, model_validator
from typing import Optional

# ---------------------------------------------------------
# 1. LOAD MODELS (IF NOT CURRENTLY IN MEMORY)
# ---------------------------------------------------------
model_id = "Qwen/Qwen2.5-7B-Instruct"

if 'tokenizer' not in globals():
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

if 'base_model' not in globals():
    print("Loading 4-bit base model (takes ~1-2 mins)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="eager"
    )
    print("Base model ready!")

if 'compressor' not in globals():
    print("Loading LLMLingua-2 compressor...")
    from llmlingua import PromptCompressor
    compressor = PromptCompressor(
        model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
        use_llmlingua2=True,
        device_map="cuda"
    )
    print("LLMLingua-2 ready!")

# ---------------------------------------------------------
# 2. DATA CONTRACTS & UTILITIES
# ---------------------------------------------------------
class CompressionRequest(BaseModel):
    context: str
    instruction: Optional[str] = Field(default="")
    rate: float = Field(default=0.5, ge=0.1, le=0.9)

class CompressionResponse(BaseModel):
    compressed_text: str
    origin_tokens: int
    compressed_tokens: int
    tokens_saved: int
    pruning_percentage: float

def compress_context(request: CompressionRequest) -> CompressionResponse:
    protected_tokens = ['\n', '?', '.', ':', '-', '"', "'"]
    results = compressor.compress_prompt(
        context=[request.context],
        instruction=request.instruction,
        rate=request.rate,
        force_tokens=protected_tokens
    )
    orig = results['origin_tokens']
    comp = results['compressed_tokens']
    saved = orig - comp
    savings_pct = round((saved / orig) * 100, 2) if orig > 0 else 0.0
    return CompressionResponse(
        compressed_text=results['compressed_prompt'],
        origin_tokens=orig,
        compressed_tokens=comp,
        tokens_saved=saved,
        pruning_percentage=savings_pct
    )

class SnapKVConfig(BaseModel):
    max_capacity: int = Field(default=512, ge=128)
    window_size: int = Field(default=32, ge=8)
    kernel_size: int = Field(default=5, ge=1)

    @model_validator(mode="after")
    def validate_capacity(self):
        if self.window_size >= self.max_capacity:
            raise ValueError("Window Size must be smaller than max_capacity")
        return self

class SnapKVManager:
    def __init__(self, model, config: SnapKVConfig):
        self.model = model
        self.config = config
        self.hooks = []
        self.num_kv_heads = model.config.num_key_value_heads
        self.num_heads = model.config.num_attention_heads
        self.num_groups = self.num_heads // self.num_kv_heads

    def _attention_hook(self, module, inputs, output):
        if not isinstance(output, tuple) or len(output) < 3 or output[2] is None:
            return output
        attention_weights = output[1]
        past_key_value = output[2]
        keys, values = past_key_value[0], past_key_value[1]

        if keys.shape[2] <= self.config.max_capacity:
            return output

        bsz = keys.shape[0]
        seq_len = attention_weights.shape[-1]
        attention_reshaped = attention_weights.view(bsz, self.num_kv_heads, self.num_groups, attention_weights.shape[2], seq_len)
        kv_attention = attention_reshaped.mean(dim=2)
        recent_attention = kv_attention[:, :, -self.config.window_size:, :-self.config.window_size]
        historical_scores = recent_attention.mean(dim=-2)
        smoothed_scores = F.avg_pool1d(historical_scores, kernel_size=self.config.kernel_size, stride=1, padding=self.config.kernel_size // 2)
        keep_historical = self.config.max_capacity - self.config.window_size

        _, top_indices = torch.topk(smoothed_scores, k=keep_historical, dim=1)
        top_indices, _ = torch.sort(top_indices, dim=-1)
        idx_expanded = top_indices.unsqueeze(-1).expand(-1, -1, -1, keys.shape[-1])

        historical_keys = keys[:, :, :-self.config.window_size, :]
        historical_values = values[:, :, :-self.config.window_size, :]

        retained_keys = torch.gather(historical_keys, 2, idx_expanded)
        retained_values = torch.gather(historical_values, 2, idx_expanded)

        protected_keys = keys[:, :, -self.config.window_size:, :]
        protected_values = values[:, :, -self.config.window_size:, :]

        new_keys = torch.cat([retained_keys, protected_keys], dim=2)
        new_values = torch.cat([retained_values, protected_values], dim=2)

        return (output[0], output[1], (new_keys, new_values))

    def turn_on(self):
        for layer in self.model.model.layers:
            hook = layer.self_attn.register_forward_hook(self._attention_hook)
            self.hooks.append(hook)

    def turn_off(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

snap_config = SnapKVConfig(max_capacity=512, window_size=32, kernel_size=5)
snapkv_engine = SnapKVManager(base_model, snap_config)

class BenchmarkRecord(BaseModel):
    configuration: str
    prompt_tokens: int
    latency_sec: float
    throughput_tps: float
    peak_vram_gb: float
    token_f1: float
    rouge_l: float

def calculate_f1_rouge(prediction: str, ground_truth: str):
    pred_tokens = prediction.strip().lower().split()
    gt_tokens = ground_truth.strip().lower().split()
    if not pred_tokens or not gt_tokens:
        return 0.0, 0.0

    common = set(pred_tokens) & set(gt_tokens)
    prec = len(common) / len(pred_tokens) if pred_tokens else 0
    rec = len(common) / len(gt_tokens) if gt_tokens else 0
    f1 = round((2 * prec * rec) / (prec + rec), 4) if (prec + rec) > 0 else 0.0

    m, n = len(pred_tokens), len(gt_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i - 1] == gt_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[m][n]
    r_prec = lcs / m if m > 0 else 0
    r_rec = lcs / n if n > 0 else 0
    rouge = round((2 * r_prec * r_rec) / (r_prec + r_rec), 4) if (r_prec + r_rec) > 0 else 0.0
    return f1, rouge

# ---------------------------------------------------------
# 3. BENCHMARK EXECUTION FUNCTION
# ---------------------------------------------------------
def run_simple_benchmarks(dataset, model, tokenizer, snapkv_manager):
    conditions = [
        {"name": "1_Baseline", "use_lingua": False, "use_snapkv": False},
        {"name": "2_LLMLingua_Only", "use_lingua": True, "use_snapkv": False},
        {"name": "3_SnapKV_Only", "use_lingua": False, "use_snapkv": True},
        {"name": "4_Integrated_Pipeline", "use_lingua": True, "use_snapkv": True}
    ]

    all_records = []

    for cond in conditions:
        print(f"\n--- Running Condition: {cond['name']} ---")

        for idx, sample in enumerate(dataset):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            context = sample["context"]
            query = sample["question"]
            ground_truth = sample["answers"]

            if cond["use_lingua"]:
                req = CompressionRequest(context=context, instruction=query, rate=0.5)
                active_context = compress_context(req).compressed_text
            else:
                active_context = context

            prompt = f"<|im_start|>system\nYou are a precise technical assistant.<|im_end|>\n<|im_start|>user\nContext:\n{active_context}\n\nQuestion: {query}<|im_end|>\n<|im_start|>assistant\n"
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            input_tokens = inputs.input_ids.shape[1]

            if cond["use_snapkv"]:
                snapkv_manager.turn_on()

            start_time = time.time()
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    do_sample=False,
                    use_cache=True
                )
            latency = time.time() - start_time

            if cond["use_snapkv"]:
                snapkv_manager.turn_off()

            generated_tokens = outputs.shape[1] - input_tokens
            peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
            throughput = generated_tokens / latency if latency > 0 else 0.0
            answer_text = tokenizer.decode(outputs[0][input_tokens:], skip_special_tokens=True)

            f1, rouge = calculate_f1_rouge(answer_text, ground_truth)

            record = BenchmarkRecord(
                configuration=cond["name"],
                prompt_tokens=input_tokens,
                latency_sec=round(latency, 3),
                throughput_tps=round(throughput, 2),
                peak_vram_gb=round(peak_vram, 2),
                token_f1=f1,
                rouge_l=rouge
            )
            all_records.append(record.model_dump())
            print(f"Sample {idx + 1} | Input: {input_tokens} toks | Latency: {round(latency, 2)}s | Peak VRAM: {round(peak_vram, 2)} GB | F1: {f1}")

    return pd.DataFrame(all_records)

# ---------------------------------------------------------
# 4. LOAD PARQUET SAMPLES & RUN
# ---------------------------------------------------------
print("Loading QASPER Parquet split...")
raw_data = load_dataset(
    "parquet",
    data_files={"test": "https://huggingface.co/datasets/allenai/qasper/resolve/refs%2Fconvert%2Fparquet/qasper/test/0000.parquet"},
    split="test"
)

test_samples = []
for paper in raw_data:
    full_paragraphs = []
    for section_name, paragraphs in zip(paper["full_text"]["section_name"], paper["full_text"]["paragraphs"]):
        for p in paragraphs:
            full_paragraphs.append(p)
    paper_context = paper["abstract"] + "\n\n" + " ".join(full_paragraphs[:12])

    for q, answers in zip(paper["qas"]["question"], paper["qas"]["answers"]):
        answer_texts = []
        for ans in answers["answer"]:
            if ans["free_form_answer"]:
                answer_texts.append(ans["free_form_answer"])
            elif ans["extractive_spans"]:
                answer_texts.extend(ans["extractive_spans"])
        if answer_texts:
            test_samples.append({
                "context": paper_context,
                "question": q,
                "answers": answer_texts[0]
            })
            break
    if len(test_samples) >= 2:
        break

print(f"\nLoaded {len(test_samples)} test samples successfully!")
results_df = run_simple_benchmarks(test_samples, base_model, tokenizer, snapkv_engine)
display(results_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Peak VRAM Footprint
sns.barplot(
    data=results_df,
    x="configuration",
    y="peak_vram_gb",
    hue="configuration",
    ax=axes[0],
    palette="Blues",
    legend=False
)
axes[0].set_title("Peak VRAM (GB) - Lower is Better", fontsize=12)
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# 2. Generation Throughput
sns.barplot(
    data=results_df,
    x="configuration",
    y="throughput_tps",
    hue="configuration",
    ax=axes[1],
    palette="Greens",
    legend=False
)
axes[1].set_title("Throughput (Tokens/Sec) - Higher is Better", fontsize=12)
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

# 3. Execution Latency
sns.barplot(
    data=results_df,
    x="configuration",
    y="latency_sec",
    hue="configuration",
    ax=axes[2],
    palette="Oranges",
    legend=False
)
axes[2].set_title("End-to-End Latency (Sec) - Lower is Better", fontsize=12)
axes[2].tick_params(axis="x", rotation=25)
axes[2].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("compression_benchmarks.png", dpi=300)
plt.show()

print("Charts successfully saved as compression_benchmarks.png!")

### **PHASE-6: GitHub Integration**
This section handles pushing the current notebook implementation to the [TokenCompression](https://github.com/ShikharVeer10/TokenCompression) repository.

In [ ]:
from google.colab import userdata
import os

# Configuration
REPO_URL = "github.com/ShikharVeer10/TokenCompression.git"
USER_NAME = "ShikharVeer10"
USER_EMAIL = "shikharveer01@gmail.com" # Replace with your actual email

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    # Construct the remote URL with the token
    remote_url = f"https://{USER_NAME}:{GITHUB_TOKEN}@{REPO_URL}"

    # Configure Git
    !git config --global user.email "{USER_EMAIL}"
    !git config --global user.name "{USER_NAME}"

    # Initialize and Push
    if not os.path.exists('.git'):
        !git init
        !git remote add origin {remote_url}

    !git add .
    !git commit -m "Update Token Compression implementation: Stage 1-3 fixed and finalized"
    !git branch -M main
    !git push -u origin main

    print("Successfully pushed to GitHub!")
except Exception as e:
    print(f"Error: {e}. Please ensure 'GITHUB_TOKEN' is set in Colab Secrets.")